### Setup

make sure FastAPI server is running in terminal `fastapi dev main.py`

In [1]:
import httpx
import time
import pandas as pd

API_URL = "http://127.0.0.1:8000/enrich"
print("Helper functions loaded. API Target:", API_URL)

Helper functions loaded. API Target: http://127.0.0.1:8000/enrich


### Test 1 - Full Pipeline (HTTP POST)

In [2]:
payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "AC01514953",
        "identifierType": "bibId",
        "gndId": "115680667",
        "wikidataId": "Q61915",
        "isbn": "3446144900",
        "fetchMarc21MD": True,
        "fetchSimilarSRU": True,
        "fetchSimilarByPerson": True,
        "fetchSimilarBySubject": True,
        "fetchSimilarByClassification": True,
        "fetchLobidGND": True,
        "fetchWikidata": True,
        "fetchCover": True,
        "fetchDescription": True,
        "maxRecs": 5
    }
}

t0 = time.perf_counter()
response = httpx.post(API_URL, json=payload, timeout=15.0)
elapsed_ms = round((time.perf_counter() - t0) * 1000, 2)

print(f"Status Code: {response.status_code} ({elapsed_ms}ms)")
assert response.status_code == 200, "API returned non-200 status code"

data = response.json()
result = data["response"]["result"]

print("Full Pipeline HTTP Test Passed!")
print("   - Title:", result["basicMarc21MD"]["title"]["titleMain"])
print("   - GND Name:", result["gndInfoLobid"]["data"]["gndInformation"]["preferredName"])
print("   - Wikidata ID:", result["wikidataData"]["data"]["wikidataId"])
print("   - Cover URL:", result["bookCover"]["data"]["coverURL"])

Status Code: 200 (5343.55ms)
Full Pipeline HTTP Test Passed!
   - Title: Schopenhauer und die wilden Jahre der Philosophie


TypeError: 'NoneType' object is not subscriptable

### Test 2 — Minimal Fetch (Fast Path)

In [3]:
payload_minimal = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "AC01514953",
        "identifierType": "ac",
        "fetchMarc21MD": True,
        "fetchLobidGND": False,
        "fetchWikidata": False,
        "fetchCover": False,
        "fetchDescription": False
    }
}

t0 = time.perf_counter()
response = httpx.post(API_URL, json=payload_minimal, timeout=10.0)
elapsed_ms = round((time.perf_counter() - t0) * 1000, 2)

assert response.status_code == 200
result = response.json()["response"]["result"]

assert result["basicMarc21MD"] is not None
assert result["gndInfoLobid"] is None
assert result["bookCover"] is None

print(f"Minimal Fetch Test Passed in {elapsed_ms}ms!")

Minimal Fetch Test Passed in 264.42ms!


### Test 3 — Direct ISBN Bypass (No MARC21 Lookup)

In [4]:
payload_bypass = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "isbn": "3446144900",
        "fetchMarc21MD": False,
        "fetchCover": True,
        "fetchDescription": True
    }
}

response = httpx.post(API_URL, json=payload_bypass, timeout=10.0)
assert response.status_code == 200
result = response.json()["response"]["result"]

assert result["basicMarc21MD"] is None
assert result["bookCover"] is not None

print("ISBN Bypass Test Passed!")
print("   - Cover URL:", result["bookCover"]["data"]["coverURL"])

ISBN Bypass Test Passed!
   - Cover URL: https://covers.openlibrary.org/b/id/2205085-M.jpg


### Test 4 — GBV Full Pipeline via PPN (Bib ID)

In [5]:
payload_gbv = {
    "iType": "bib",
    "institution": {
        "iName": "gbv",
        "identifier": "1977213316",
        "identifierType": "bibId",
        "fetchMarc21MD": True,
        "fetchSimilarSRU": True,
        "fetchSimilarByPerson": True,
        "fetchSimilarBySubject": True,
        "fetchSimilarByClassification": True,
        "fetchLobidGND": True,
        "fetchWikidata": True,
        "fetchCover": True,
        "fetchDescription": True,
        "maxRecs": 5
    }
}

t0 = time.perf_counter()
response = httpx.post(API_URL, json=payload_gbv, timeout=15.0)
elapsed_ms = round((time.perf_counter() - t0) * 1000, 2)

print(f"Status Code: {response.status_code} ({elapsed_ms}ms)")
assert response.status_code == 200, "API returned non-200 status code for GBV"

data = response.json()
result = data["response"]["result"]

print("GBV PPN Pipeline Test Passed!")
print("   - Institution Name:", result["iName"])
print("   - Title:", result["basicMarc21MD"]["title"]["titleMain"])
print("   - Primary PPN (001):", result["basicMarc21MD"]["primaryBibIdentifier"])
print("   - Similar Records (Person):", len(result["additionalRecsSRU"]["records"][1]["additionalRecs"]))

Status Code: 200 (4642.79ms)
GBV PPN Pipeline Test Passed!
   - Institution Name: gbv
   - Title: En åndeseers drømme
   - Primary PPN (001): 1977213316
   - Similar Records (Person): 5


### Test 5 — GBV & OeNB Lookup via ISBN Identifier Type

In [6]:
# Test GBV via ISBN lookup
payload_gbv_isbn = {
    "iType": "bib",
    "institution": {
        "iName": "gbv",
        "identifier": "9783499007408",
        "identifierType": "isbn",
        "fetchMarc21MD": True,
        "fetchSimilarSRU": False,
        "fetchCover": True
    }
}

res_gbv = httpx.post(API_URL, json=payload_gbv_isbn, timeout=15.0)
assert res_gbv.status_code == 200
r_gbv_data = res_gbv.json()["response"]["result"]

# Test OeNB via ISBN lookup
payload_oenb_isbn = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "9783209115393",
        "identifierType": "isbn",
        "fetchMarc21MD": True,
        "fetchSimilarSRU": False,
        "fetchCover": True
    }
}

res_oenb = httpx.post(API_URL, json=payload_oenb_isbn, timeout=15.0)
assert res_oenb.status_code == 200
r_oenb_data = res_oenb.json()["response"]["result"]

print("ISBN Identifier Type Tests Passed Successfully!")
print("   - GBV ISBN Title:", r_gbv_data["basicMarc21MD"]["title"]["titleMain"])
print("   - OeNB ISBN Title:", r_oenb_data["basicMarc21MD"]["title"]["titleMain"])

ISBN Identifier Type Tests Passed Successfully!
   - GBV ISBN Title: Sorgt, dass sie nicht zu zeitig mich erwecken
   - OeNB ISBN Title: BioTOP


### Quick Response Time Benchmark Matrix

In [ ]:
scenarios = [
    ("Full Pipeline", True, True, True, True, True),
    ("MARC21 Only", True, False, False, False, False),
    ("MARC21 + GND/Wikidata", True, False, True, True, False),
    ("MARC21 + Cover/Desc", True, False, False, False, True),
]

auditResults = []

for name, marc, sru, lobid, wiki, media in scenarios:
    p = {
        "iType": "bib",
        "institution": {
            "iName": "oenb",
            "identifier": "AC01514953",
            "identifierType": "ac",
            "fetchMarc21MD": marc,
            "fetchLobidGND": lobid,
            "fetchWikidata": wiki,
            "fetchCover": media,
            "fetchDescription": media
        }
    }
    t0 = time.perf_counter()
    r = httpx.post(API_URL, json=p, timeout=15.0)
    dur = round((time.perf_counter() - t0) * 1000, 2)
    
    auditResults.append({
        "Scenario": name,
        "HTTP Status": r.status_code,
        "Total Time (ms)": dur
    })

pd.DataFrame(auditResults)

,Scenario,HTTP Status,Total Time (ms)
0,Full Pipeline,200,4295.77
1,MARC21 Only,200,252.41
2,MARC21 + GND/Wikidata,200,532.52
3,MARC21 + Cover/Desc,200,4244.63
